# Per-User Resource Cleanup

Scope (extends the `Databricks Cost Tracking.lvdash.json` dashboard):
clusters, jobs, DLT pipelines, SQL warehouses, model serving endpoints,
Databricks Apps, Vector Search endpoints, Lakebase database instances,
and Unity Catalog catalogs.

| Source | "Created date" | "Last used" |
| --- | --- | --- |
| clusters / jobs / pipelines / warehouses | `MIN(change_time)` from system table | `MAX(usage_date)` from `system.billing.usage` |
| serving endpoints / apps | SDK `creation_timestamp` / `create_time` | `MAX(usage_date)` for `endpoint_id` / `app_id` |
| vector search / lakebase / catalogs | SDK `creation_timestamp` / `creation_time` / `created_at` | not tracked — age-only |

A resource is a candidate when:
`created < today - max_age_days` AND `(last_used signal absent OR last_used < today - last_used_lookback_days)`
AND owner matches `owner_filter` AND `exclude_tag` is not set.

**Always run with `dry_run=true` first.**

In [0]:
dbutils.widgets.text("max_age_days", "30", "Created more than N days ago")
dbutils.widgets.text("last_used_lookback_days", "14", "Last billed within N days = keep")
dbutils.widgets.text("owner_filter", "", "Owners (comma-sep emails/SPs/regex). Empty = all owners")
dbutils.widgets.text(
    "resource_types",
    "clusters,jobs,pipelines,warehouses,serving,apps,vector_search,lakebase,catalogs",
    "Comma-sep types to include",
)
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
dbutils.widgets.text("exclude_tag", "NoAutoUpdate", "Skip resources with this tag key (any value)")

MAX_AGE_DAYS            = int(dbutils.widgets.get("max_age_days"))
LAST_USED_LOOKBACK_DAYS = int(dbutils.widgets.get("last_used_lookback_days"))
OWNER_FILTER_RAW        = dbutils.widgets.get("owner_filter").strip()
RESOURCE_TYPES_RAW      = dbutils.widgets.get("resource_types").strip()
DRY_RUN                 = dbutils.widgets.get("dry_run").lower() == "true"
EXCLUDE_TAG             = dbutils.widgets.get("exclude_tag").strip()

owner_list     = [o.strip() for o in OWNER_FILTER_RAW.split(",") if o.strip()]
resource_types = {t.strip() for t in RESOURCE_TYPES_RAW.split(",") if t.strip()}

CURRENT_WORKSPACE_ID = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().workspaceId().get()
)

print(f"workspace_id={CURRENT_WORKSPACE_ID} (pinned to current workspace)")
print(f"max_age_days={MAX_AGE_DAYS}  last_used_lookback={LAST_USED_LOOKBACK_DAYS}")
print(f"owner_filter={owner_list or 'ALL'}")
print(f"resource_types={sorted(resource_types)}")
print(f"dry_run={DRY_RUN}  exclude_tag={EXCLUDE_TAG!r}")

workspace_id=984752964297111 (pinned to current workspace)
max_age_days=30  last_used_lookback=14
owner_filter=['steve.shao@databricks.com']
resource_types=['apps', 'catalogs', 'clusters', 'jobs', 'lakebase', 'pipelines', 'serving', 'vector_search', 'warehouses']
dry_run=True  exclude_tag='NoAutoUpdate'


In [0]:
import re
import datetime as _dt
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DateType, MapType,
)

UNIFIED_SCHEMA = StructType([
    StructField("resource_type",   StringType(),  False),
    StructField("workspace_id",    StringType(),  True),
    StructField("resource_id",     StringType(),  False),
    StructField("resource_name",   StringType(),  True),
    StructField("owner",           StringType(),  True),
    StructField("created_time",    TimestampType(), True),
    StructField("last_used_date",  DateType(),    True),
    StructField("tags",            MapType(StringType(), StringType()), True),
])

def owner_predicate(col):
    if not owner_list:
        return F.lit(True)
    literals = [o for o in owner_list if not o.startswith("re:")]
    regexes  = [o[3:] for o in owner_list if o.startswith("re:")]
    pred = F.lit(False)
    if literals:
        pred = pred | col.isin(literals)
    for rx in regexes:
        pred = pred | col.rlike(rx)
    return pred

def owner_matches_py(owner: str) -> bool:
    if not owner_list:
        return True
    if owner is None:
        return False
    for o in owner_list:
        if o.startswith("re:"):
            if re.search(o[3:], owner):
                return True
        elif owner == o:
            return True
    return False

def has_exclude_tag(tags_col):
    if not EXCLUDE_TAG:
        return F.lit(False)
    return F.coalesce(F.element_at(tags_col, EXCLUDE_TAG).isNotNull(), F.lit(False))

def has_exclude_tag_py(tags: dict) -> bool:
    return bool(EXCLUDE_TAG) and EXCLUDE_TAG in (tags or {})

age_cutoff = F.expr(f"current_date() - INTERVAL {MAX_AGE_DAYS} DAYS")
age_cutoff_dt = _dt.datetime.utcnow() - _dt.timedelta(days=MAX_AGE_DAYS)
last_used_cutoff_dt = _dt.date.today() - _dt.timedelta(days=LAST_USED_LOOKBACK_DAYS)

# Cached SDK client — used by SDK-native sources and the delete loop.
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, ResourceDoesNotExist, PermissionDenied
w = WorkspaceClient()

## System-table sources (clusters / jobs / pipelines / warehouses)

In [0]:
usage = spark.table("system.billing.usage").filter(
    F.col("usage_date") >= F.expr(f"current_date() - INTERVAL {LAST_USED_LOOKBACK_DAYS} DAYS")
)

def _ws_filter(df):
    return df.filter(F.col("workspace_id") == F.lit(CURRENT_WORKSPACE_ID))

def _last_used_by(usage_id_col, *, billing_filter=None):
    df = _ws_filter(usage)
    if billing_filter is not None:
        df = df.filter(billing_filter)
    return (df
        .filter(F.col(usage_id_col).isNotNull())
        .groupBy(F.col(usage_id_col).alias("rid"))
        .agg(F.max("usage_date").alias("last_used_date"))
    )

def _systable_meta(table, id_col, *, name_col, owner_col):
    return (_ws_filter(spark.table(table))
        .groupBy("workspace_id", id_col)
        .agg(
            F.min("change_time").alias("created_time"),
            F.max_by(name_col,  "change_time").alias("resource_name"),
            F.max_by(owner_col, "change_time").alias("owner"),
            F.max_by("delete_time", "change_time").alias("delete_time"),
            F.max_by("tags", "change_time").alias("tags"),
        )
    )

def _build_systable_candidates(rtype, table, id_col, *, name_col, owner_col, last_used_df):
    return (_systable_meta(table, id_col, name_col=name_col, owner_col=owner_col)
        .join(last_used_df, F.col(id_col) == F.col("rid"), "left").drop("rid")
        .filter(F.col("delete_time").isNull())
        .filter(F.col("created_time") < age_cutoff)
        .filter(F.col("last_used_date").isNull())
        .filter(owner_predicate(F.col("owner")))
        .filter(~has_exclude_tag(F.col("tags")))
        .withColumn("resource_type", F.lit(rtype))
        .select(
            "resource_type",
            F.col("workspace_id").cast("string").alias("workspace_id"),
            F.col(id_col).cast("string").alias("resource_id"),
            "resource_name",
            "owner",
            "created_time",
            "last_used_date",
            "tags",
        )
    )

systable_dfs = {}

if "clusters" in resource_types:
    systable_dfs["clusters"] = _build_systable_candidates(
        "cluster", "system.compute.clusters", "cluster_id",
        name_col="cluster_name", owner_col="owned_by",
        last_used_df=_last_used_by(
            "usage_metadata.cluster_id",
            billing_filter=F.col("billing_origin_product").isin("ALL_PURPOSE", "INTERACTIVE"),
        ),
    )

if "jobs" in resource_types:
    # Jobs use COALESCE(run_as, creator_id) as owner.
    jobs = (_ws_filter(spark.table("system.lakeflow.jobs"))
        .groupBy("workspace_id", "job_id")
        .agg(
            F.min("change_time").alias("created_time"),
            F.max_by("name",        "change_time").alias("resource_name"),
            F.max_by("creator_id",  "change_time").alias("creator_id"),
            F.max_by("run_as",      "change_time").alias("run_as"),
            F.max_by("delete_time", "change_time").alias("delete_time"),
            F.max_by("tags",        "change_time").alias("tags"),
        )
        .withColumn("owner", F.coalesce(F.col("run_as"), F.col("creator_id")))
    )
    systable_dfs["jobs"] = (jobs
        .join(_last_used_by("usage_metadata.job_id",
                            billing_filter=F.col("billing_origin_product") == "JOBS"),
              F.col("job_id") == F.col("rid"), "left").drop("rid")
        .filter(F.col("delete_time").isNull())
        .filter(F.col("created_time") < age_cutoff)
        .filter(F.col("last_used_date").isNull())
        .filter(owner_predicate(F.col("owner")))
        .filter(~has_exclude_tag(F.col("tags")))
        .withColumn("resource_type", F.lit("job"))
        .select(
            "resource_type",
            F.col("workspace_id").cast("string").alias("workspace_id"),
            F.col("job_id").cast("string").alias("resource_id"),
            "resource_name", "owner", "created_time", "last_used_date", "tags",
        )
    )

if "pipelines" in resource_types:
    # DLT pipelines from system.lakeflow.pipelines.
    pipelines = (_ws_filter(spark.table("system.lakeflow.pipelines"))
        .groupBy("workspace_id", "pipeline_id")
        .agg(
            F.min("change_time").alias("created_time"),
            F.max_by("name",        "change_time").alias("resource_name"),
            F.max_by("created_by",  "change_time").alias("created_by"),
            F.max_by("run_as",      "change_time").alias("run_as"),
            F.max_by("delete_time", "change_time").alias("delete_time"),
            F.max_by("tags",        "change_time").alias("tags"),
        )
        .withColumn("owner", F.coalesce(F.col("run_as"), F.col("created_by")))
    )
    systable_dfs["pipelines"] = (pipelines
        .join(_last_used_by("usage_metadata.dlt_pipeline_id"),
              F.col("pipeline_id") == F.col("rid"), "left").drop("rid")
        .filter(F.col("delete_time").isNull())
        .filter(F.col("created_time") < age_cutoff)
        .filter(F.col("last_used_date").isNull())
        .filter(owner_predicate(F.col("owner")))
        .filter(~has_exclude_tag(F.col("tags")))
        .withColumn("resource_type", F.lit("pipeline"))
        .select(
            "resource_type",
            F.col("workspace_id").cast("string").alias("workspace_id"),
            F.col("pipeline_id").cast("string").alias("resource_id"),
            "resource_name", "owner", "created_time", "last_used_date", "tags",
        )
    )

if "warehouses" in resource_types:
    systable_dfs["warehouses"] = _build_systable_candidates(
        "warehouse", "system.compute.warehouses", "warehouse_id",
        name_col="warehouse_name", owner_col="created_by",
        last_used_df=_last_used_by(
            "usage_metadata.warehouse_id",
            billing_filter=F.col("billing_origin_product") == "SQL",
        ),
    )

## SDK-native sources (apps / serving / vector search / lakebase / catalogs)

In [0]:
def _epoch_to_ts(v):
    """Accepts datetime, epoch-ms int, or None. Returns datetime or None."""
    if v is None:
        return None
    if isinstance(v, _dt.datetime):
        return v
    try:
        return _dt.datetime.utcfromtimestamp(int(v) / 1000.0)
    except Exception:
        return None

def _last_used_by_id_set(usage_id_col, billing_filter=None):
    """Returns dict: resource_id (str) -> last_used_date (date)."""
    df = _ws_filter(usage)
    if billing_filter is not None:
        df = df.filter(billing_filter)
    rows = (df.filter(F.col(usage_id_col).isNotNull())
              .groupBy(F.col(usage_id_col).cast("string").alias("rid"))
              .agg(F.max("usage_date").alias("last_used_date"))
              .collect())
    return {r["rid"]: r["last_used_date"] for r in rows}

def _row(rtype, rid, name, owner, created_time, last_used_date=None, tags=None):
    return (rtype, str(CURRENT_WORKSPACE_ID), str(rid), name, owner,
            created_time, last_used_date, dict(tags or {}))

def _passes_filters(created_time, owner, tags, last_used_date):
    if created_time is None or created_time >= age_cutoff_dt:
        return False
    if last_used_date is not None and last_used_date >= last_used_cutoff_dt:
        return False
    if not owner_matches_py(owner):
        return False
    if has_exclude_tag_py(tags):
        return False
    return True

def _safe_iter(label, fn):
    """Run an SDK list call, swallow auth/feature-disabled errors."""
    try:
        return list(fn())
    except (PermissionDenied, NotFound) as e:
        print(f"[{label}] skipped: {e}")
        return []
    except Exception as e:
        print(f"[{label}] error: {e}")
        return []

sdk_rows = []

# --- Databricks Apps ---
if "apps" in resource_types:
    apps_last_used = _last_used_by_id_set("usage_metadata.app_id")
    for a in _safe_iter("apps", lambda: w.apps.list()):
        created = _epoch_to_ts(getattr(a, "create_time", None))
        owner   = getattr(a, "creator", None)
        tags    = {}
        last    = apps_last_used.get(getattr(a, "name", "") or "")
        if _passes_filters(created, owner, tags, last):
            sdk_rows.append(_row("app", getattr(a, "name", None),
                                 getattr(a, "name", None), owner, created, last, tags))

# --- Model Serving Endpoints ---
if "serving" in resource_types:
    serving_last_used = _last_used_by_id_set("usage_metadata.endpoint_id")
    for ep in _safe_iter("serving", lambda: w.serving_endpoints.list()):
        created = _epoch_to_ts(getattr(ep, "creation_timestamp", None))
        owner   = getattr(ep, "creator", None)
        tag_list = getattr(ep, "tags", None) or []
        tags = {}
        for t in tag_list:
            k = getattr(t, "key", None); v = getattr(t, "value", None)
            if k: tags[k] = v
        rid = getattr(ep, "id", None) or getattr(ep, "name", None)
        last = serving_last_used.get(str(rid) if rid is not None else "")
        if _passes_filters(created, owner, tags, last):
            sdk_rows.append(_row("serving_endpoint", rid,
                                 getattr(ep, "name", None), owner, created, last, tags))

# --- Vector Search Endpoints ---
if "vector_search" in resource_types:
    for ep in _safe_iter("vector_search", lambda: w.vector_search_endpoints.list_endpoints()):
        created = _epoch_to_ts(getattr(ep, "creation_timestamp", None))
        owner   = getattr(ep, "creator", None)
        rid     = getattr(ep, "id", None) or getattr(ep, "name", None)
        if _passes_filters(created, owner, {}, None):
            sdk_rows.append(_row("vs_endpoint", rid,
                                 getattr(ep, "name", None), owner, created, None, {}))

# --- Lakebase Database Instances ---
if "lakebase" in resource_types:
    def _list_lakebase():
        # SDK surface differs across versions; try the current canonical path first.
        api = getattr(w, "database", None)
        if api is None:
            return []
        for fn_name in ("list_database_instances", "list_instances"):
            fn = getattr(api, fn_name, None)
            if fn is not None:
                return fn()
        return []
    for inst in _safe_iter("lakebase", _list_lakebase):
        created = getattr(inst, "creation_time", None)
        if not isinstance(created, _dt.datetime):
            created = _epoch_to_ts(created)
        owner = (getattr(inst, "creator", None)
                 or getattr(inst, "creator_user_name", None)
                 or getattr(inst, "owner", None))
        rid   = getattr(inst, "uid", None) or getattr(inst, "name", None)
        if _passes_filters(created, owner, {}, None):
            sdk_rows.append(_row("lakebase_instance", rid,
                                 getattr(inst, "name", None), owner, created, None, {}))

# --- Unity Catalog catalogs ---
# Catalogs are metastore-scoped, not workspace-scoped — workspace_id column will
# be the current workspace for bookkeeping only. Foreign / system catalogs are
# excluded so we don't try to delete them.
if "catalogs" in resource_types:
    for c in _safe_iter("catalogs", lambda: w.catalogs.list()):
        ctype = getattr(c, "catalog_type", None)
        ctype_str = getattr(ctype, "value", None) or str(ctype) if ctype else ""
        if "FOREIGN" in ctype_str or "SYSTEM" in ctype_str:
            continue
        created = _epoch_to_ts(getattr(c, "created_at", None))
        owner   = getattr(c, "owner", None)
        tags    = dict(getattr(c, "properties", None) or {})
        rid     = getattr(c, "name", None)
        if _passes_filters(created, owner, tags, None):
            sdk_rows.append(_row("catalog", rid, rid, owner, created, None, tags))

if sdk_rows:
    sdk_df = spark.createDataFrame(sdk_rows, schema=UNIFIED_SCHEMA)
else:
    sdk_df = spark.createDataFrame([], schema=UNIFIED_SCHEMA)

[apps] error: 'AppsAPI' object has no attribute 'list'


## Combined candidate list

In [0]:
candidate_dfs = list(systable_dfs.values()) + ([sdk_df] if sdk_rows else [])
if not candidate_dfs:
    print("No candidate sources selected.")
    candidates = spark.createDataFrame([], schema=UNIFIED_SCHEMA)
else:
    candidates = candidate_dfs[0]
    for d in candidate_dfs[1:]:
        candidates = candidates.unionByName(d)

print(f"candidate count: {candidates.count()}")
display(candidates.orderBy("resource_type", "owner", "created_time"))

candidate count: 9


resource_type,workspace_id,resource_id,resource_name,owner,created_time,last_used_date,tags
catalog,984752964297111,steve_share,steve_share,steve.shao@databricks.com,2025-12-04T02:53:04.679Z,null,Map()
catalog,984752964297111,haier_eu_share_test,haier_eu_share_test,steve.shao@databricks.com,2025-12-15T03:15:51.785Z,null,Map()
catalog,984752964297111,steve_patient_sample,steve_patient_sample,steve.shao@databricks.com,2025-12-17T00:50:37.171Z,null,Map()
catalog,984752964297111,steve_shao,steve_shao,steve.shao@databricks.com,2025-12-24T00:00:54.135Z,null,Map()
catalog,984752964297111,steve_sqlserver_test,steve_sqlserver_test,steve.shao@databricks.com,2025-12-24T08:12:59.740Z,null,Map()
catalog,984752964297111,shao_sandbox1_pg,shao_sandbox1_pg,steve.shao@databricks.com,2026-01-07T06:28:33.172Z,null,"Map(endpoint_id -> 6587c340-644a-4e6f-92da-23408fde057e, workspace_id -> 984752964297111)"
catalog,984752964297111,steve_deltasharing_demo,steve_deltasharing_demo,steve.shao@databricks.com,2026-01-23T02:18:53.293Z,null,Map()
catalog,984752964297111,lenovo-genie,lenovo-genie,steve.shao@databricks.com,2026-03-20T00:07:20.858Z,null,Map()
pipeline,984752964297111,8e4a9cda-48f3-4ec9-a1a4-a4c6fae09c40,steve-salesforce,steve.shao@databricks.com,2026-02-10T02:12:56.266Z,null,Map()


## Delete (only when `dry_run=false`)
Acts via SDK against the current workspace and the metastore attached to it.
Catalogs are deleted **without** force — schemas must be empty.

In [0]:
if DRY_RUN:
    print("DRY RUN — no resources deleted. Set dry_run=false to actually delete.")
    dbutils.notebook.exit("dry_run")

def _delete_one(rtype: str, rid: str, rname: str):
    if rtype == "cluster":
        w.clusters.permanent_delete(cluster_id=rid)
    elif rtype == "job":
        w.jobs.delete(job_id=int(rid))
    elif rtype == "pipeline":
        w.pipelines.delete(pipeline_id=rid)
    elif rtype == "warehouse":
        w.warehouses.delete(id=rid)
    elif rtype == "serving_endpoint":
        w.serving_endpoints.delete(name=rname)
    elif rtype == "app":
        w.apps.delete(name=rname)
    elif rtype == "vs_endpoint":
        w.vector_search_endpoints.delete_endpoint(endpoint_name=rname)
    elif rtype == "lakebase_instance":
        api = getattr(w, "database", None)
        for fn_name in ("delete_database_instance", "delete_instance"):
            fn = getattr(api, fn_name, None) if api else None
            if fn:
                # purge=False keeps storage retained per Lakebase soft-delete window
                try:    fn(name=rname, purge=False)
                except TypeError: fn(name=rname)
                return
        raise RuntimeError("No Lakebase delete API found on this SDK")
    elif rtype == "catalog":
        w.catalogs.delete(name=rname, force=False)
    else:
        raise ValueError(f"unknown resource_type {rtype}")

results = []
for r in candidates.collect():
    rid, rtype, rname, rowner = r["resource_id"], r["resource_type"], r["resource_name"], r["owner"]
    try:
        _delete_one(rtype, rid, rname)
        results.append((rtype, rid, rname, rowner, "deleted", None))
    except (NotFound, ResourceDoesNotExist) as e:
        results.append((rtype, rid, rname, rowner, "not_found", str(e)))
    except Exception as e:
        results.append((rtype, rid, rname, rowner, "error", str(e)))

result_df = spark.createDataFrame(
    results,
    "resource_type STRING, resource_id STRING, resource_name STRING, owner STRING, status STRING, error STRING",
)
display(result_df)

# Optional: persist an audit log
# (result_df.withColumn("run_ts", F.current_timestamp())
#           .write.mode("append").saveAsTable("main.ops.resource_cleanup_audit"))